# Cosmos3-Edge Policy with TensorRT-Edge-LLM

This notebook runs **Cosmos3-Edge-Policy-DROID** with
[TensorRT-Edge-LLM](https://github.com/NVIDIA/TensorRT-Edge-LLM) (v0.10.0+):
export the policy checkpoint, build the experimental Cosmos3 policy engines,
then run `cosmos3_policy_inference`.

Policy sits under the **Generator / Action** cookbook surface (action chunks),
not the Reasoner text path. For Edge multimodal reasoning on the same runtime,
see [`../../reasoner/run_with_trt_edge_llm.ipynb`](../../reasoner/run_with_trt_edge_llm.ipynb).

1. Points at a TensorRT-Edge-LLM checkout built with
   `-DBUILD_EXPERIMENTAL_MODELS=ON`.
2. Exports `nvidia/Cosmos3-Edge-Policy-DROID` with `--task policy`.
3. Builds policy engines.
4. Runs a single-image policy request and writes `action.json`.

Complete the upstream
[Installation](https://nvidia.github.io/TensorRT-Edge-LLM/user_guide/getting_started/installation.html)
first. Command reference:
[Cosmos3-Edge VLA guide](https://nvidia.github.io/TensorRT-Edge-LLM/user_guide/examples/vla/cosmos3.html).


## 1. Setup paths

`EDGELLM_ROOT` must contain `build/experimental_models/cosmos3/examples/` after
a CMake build with `-DBUILD_EXPERIMENTAL_MODELS=ON`.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "cookbooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the cosmos repository root")


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
# Reuse the reasoner robot image as a simple single-frame observation.
OBSERVATION_IMAGE = (
    COSMOS_ROOT / "cookbooks" / "cosmos3" / "reasoner" / "assets" / "robot_153.jpg"
).resolve()
assert OBSERVATION_IMAGE.exists(), OBSERVATION_IMAGE

EDGELLM_ROOT = Path(
    os.environ.get("EDGELLM_ROOT", Path.home() / "TensorRT-Edge-LLM")
).expanduser().resolve()
ONNX_DIR = Path(
    os.environ.get(
        "ONNX_DIR",
        Path.home() / "tensorrt-edgellm-workspace" / "Cosmos3-Edge" / "onnx",
    )
).expanduser().resolve()
ENGINE_DIR = Path(
    os.environ.get(
        "ENGINE_DIR",
        Path.home() / "tensorrt-edgellm-workspace" / "Cosmos3-Edge" / "engines",
    )
).expanduser().resolve()
POLICY_CHECKPOINT = os.environ.get(
    "POLICY_CHECKPOINT", "nvidia/Cosmos3-Edge-Policy-DROID"
)
POLICY_PROMPT = os.environ.get(
    "POLICY_PROMPT",
    "Pick up the banana and place it in the bowl.",
)
WORK_DIR = ENGINE_DIR / "policy" / "cookbook_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON = WORK_DIR / "action.json"

os.environ["COSMOS_ROOT"] = str(COSMOS_ROOT)
os.environ["EDGELLM_ROOT"] = str(EDGELLM_ROOT)
os.environ["ONNX_DIR"] = str(ONNX_DIR)
os.environ["ENGINE_DIR"] = str(ENGINE_DIR)
os.environ["POLICY_CHECKPOINT"] = POLICY_CHECKPOINT
os.environ["OBSERVATION_IMAGE"] = str(OBSERVATION_IMAGE)
os.environ["POLICY_PROMPT"] = POLICY_PROMPT
os.environ["OUTPUT_JSON"] = str(OUTPUT_JSON)

print("cosmos root:", COSMOS_ROOT)
print("EDGELLM_ROOT:", EDGELLM_ROOT, "(exists:", EDGELLM_ROOT.exists(), ")")
print("ONNX_DIR:", ONNX_DIR)
print("ENGINE_DIR:", ENGINE_DIR)
print("POLICY_CHECKPOINT:", POLICY_CHECKPOINT)
print("observation:", OBSERVATION_IMAGE)
print("prompt:", POLICY_PROMPT)


## 2. Export the Policy on CPU

Requires the TensorRT-Edge-LLM Python package (`tensorrt-edgellm-export`).


In [ ]:
%%bash
set -euo pipefail
: "${POLICY_CHECKPOINT:?run the setup cell first}"
: "${ONNX_DIR:?run the setup cell first}"

mkdir -p "$ONNX_DIR"
tensorrt-edgellm-export \
  "$POLICY_CHECKPOINT" \
  "$ONNX_DIR" \
  --task policy

echo "Exported policy ONNX under $ONNX_DIR"


## 3. Build all policy engines

The C++ runtime must have been configured with
`-DBUILD_EXPERIMENTAL_MODELS=ON`. Use `--maxBatchSize N` when the runtime must
accept more than one prompt.


In [ ]:
%%bash
set -euo pipefail
: "${EDGELLM_ROOT:?run the setup cell first}"
: "${ONNX_DIR:?run the setup cell first}"
: "${ENGINE_DIR:?run the setup cell first}"

cd "$EDGELLM_ROOT"
mkdir -p "$ENGINE_DIR"

./build/experimental_models/cosmos3/examples/cosmos3_policy_build \
  --onnxDir "$ONNX_DIR" \
  --engineDir "$ENGINE_DIR"

echo "Built policy engines under $ENGINE_DIR"


## 4. Run the policy

Single observation image. For an ordered frame list, replace `--image` with
`--video frame_00.png,frame_01.png,frame_02.png`. The current policy conditions
on the most recent frame. Optional: `--steps` (denoise steps) and `--seed`.


In [ ]:
%%bash
set -euo pipefail
: "${EDGELLM_ROOT:?run the setup cell first}"
: "${ENGINE_DIR:?run the setup cell first}"
: "${OBSERVATION_IMAGE:?run the setup cell first}"
: "${POLICY_PROMPT:?run the setup cell first}"
: "${OUTPUT_JSON:?run the setup cell first}"

cd "$EDGELLM_ROOT"
./build/experimental_models/cosmos3/examples/cosmos3_policy_inference \
  --engineDir "$ENGINE_DIR" \
  --image "$OBSERVATION_IMAGE" \
  --prompt "$POLICY_PROMPT" \
  --output "$OUTPUT_JSON"

echo "Wrote $OUTPUT_JSON"
python3 - <<'PY'
import json
import os
from pathlib import Path
data = json.loads(Path(os.environ["OUTPUT_JSON"]).read_text())
print(json.dumps(data, indent=2)[:4000])
PY


## 5. Next steps

- Override `POLICY_PROMPT` / `OBSERVATION_IMAGE` in the setup cell for your
  DROID observation.
- Multimodal reasoning on TensorRT-Edge-LLM:
  [`../../reasoner/run_with_trt_edge_llm.ipynb`](../../reasoner/run_with_trt_edge_llm.ipynb).
- OpenAI-compatible policy serving on workstation GPUs:
  [`run_policy_with_vllm_omni.ipynb`](./run_policy_with_vllm_omni.ipynb).
- Cosmos Framework policy server:
  [`run_policy_with_cosmos_framework.md`](./run_policy_with_cosmos_framework.md).
